# INSTALLING AND IMPORTING DEPENDENCIES

In [ ]:
pip install gymnasium

In [ ]:
pip install optuna

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import optuna
import gymnasium as gym
from collections import deque

In [ ]:
with open("model.pkl","rb") as f:
  model = pickle.load(f)

In [ ]:
with open("others.pkl","rb") as f:
  others = pickle.load(f)

In [ ]:
with open("quantile.pkl","rb") as f:
  quantile = pickle.load(f)

In [ ]:
# test.csv - Dataset I created by combining the actual train data,
# the predictions of my model, and the 10th percentile prediciton of the quantile model

df = pd.read_csv("./test.csv")

# ENVIRONMENT

To train a bidding agent - I create a custom environment in **Gymnasium** library. This environment simulates an auction similar to the Arena my agent will be participating in. The opponents of this environment are drawn from a normal distribution centered around 0 and $\text{variance} = 0.1 \times \text{volatility}$ of my predicitons.

The volatility I defined is the difference between the predicted price by my model ($P_{average}$) and the 10th percentile price predicted by my quantile model ($P_{10}$). This helps my agent to understand the uncertainity and risk in bidding against a car with volatile prices.

The agent wins if it's bid is more than it's rival price generated by the normal distribution. The opponents price is drawn from a normal distribution so as to stimulate an enviornment where all kinds of bids within reasonable boundaries are possible . This makes the agent learn the different kinds of bids it may face and accordingly judge when to bid and when to fold.

The agent profits if it makes a successfull purchase, that is, profits on a car. If it does not make a profit, it is penalized. If the agent goes bankrupt, it is heavily penalized so that agent learns to manage the bankroll appropriately.

In [ ]:
class CarAuctionEnv(gym.Env):

  def __init__(self,df):
    super().__init__()
    self.cars = 500
    self.dataset = df
    self.bankroll = 500000

  def reset(self, seed=None, options=None):
    super().reset(seed=seed)
    self.bankroll = 500000
    self.current_car_index = 0
    self.recent_wins = deque(maxlen=50)

    self.dataset = self.dataset.sample(frac=1).reset_index(drop=True)
    return self._get_obs(), {}

  def _get_obs(self):
    row = self.dataset.iloc[self.current_car_index]

    price = row["pred"]
    volatility = row["std"]
    win_rate = sum(self.recent_wins) / len(self.recent_wins) if len(self.recent_wins) > 0 else 0.0

    return np.array([price,volatility,self.bankroll, self.current_car_index + 1, win_rate], dtype=np.float32)

  def step(self, agent_bid):
    row = self.dataset.iloc[self.current_car_index]
    selling_price = row["sellingprice"]

    noise = np.random.normal(loc=0, scale=selling_price*0.1)
    rival_bid = selling_price + noise

    profit = 0
    won_auction = False

    if agent_bid > rival_bid:
      won_auction = True
      price_paid = rival_bid + 50.0

      if price_paid <= self.bankroll:
        profit = selling_price - price_paid
        self.bankroll += profit

      else:
        won_auction = False
        profit = -self.bankroll
        self.bankroll = 0

    self.recent_wins.append(won_auction)
    self.current_car_index += 1

    terminated = bool((self.bankroll <= 0) or (self.current_car_index >= self.cars))

    reward = profit

    info = {
        "won":won_auction,
        "true_price":selling_price,
        "agent_bid":agent_bid,
        "rival_bid":rival_bid,
        "profit":profit
    }

    return self._get_obs(), reward, terminated, False, info

# OPTIMIZATION

The agent bids according to this formula:

Let Predicted Price be $\mu$. Then,

$\text{Bid} = \mu - (\alpha \times \text{volatility}) - \beta \times \frac{500,000 - \text{bankroll}}{500,000} \times \mu + \gamma \times \text{win} \times \mu$

The formula is derived as follows:

$(\mu - \alpha \times \text{volatility})$: This represents the base price of my agent. The volatility terms acts as a measure of risk/uncertainity in the price of the car.
Consider a car with predicted price 15k. If the volatility is 1k, then even in worse conditions (bottom 10 percent), the car's actual price for 14k. If we bid 15k, we can lose about 1k, minimizing our risk. The downside risk of that car is low.
However if the volatility is 5k, then in worse conditions(again bottom 10 percent), the car may sell for as low as 10k. If we bid 15k on the car, we may lose upto 5k. The downside risk of that is high
If the car is more volatile ($P_{average} - P_{50}$ is higher), then the formula adjusts the bid such that it lowers the bidding amount. $\alpha$ is the weight given to the volatility.

$(\beta \times \frac{500,000 - \text{bankroll}}{500,000} \times \mu)$: This is our penalty. The ratio $\frac{500000 - \text{bankroll}}{500000}$ gives the measure of our current bankroll. If it is low, our agent is losing money and needs to restrict itself to be conservative to conserve capital. Thus it lowers it's bid. $\beta$ is the weight given to the penalty.

 $(\gamma \times \text{win} \times \mu)$: This is the boost given to bid. Our current formula restricts our agent to be highly selective and conservative, bidding only for cars it is sure and even then a lower amount. However this may make the agent inactive for the whole auction. To improve this, a boost is given to the bid. This makes our agent bid a higher price than it's prediction to win some cars, even if it means to make less profit. It is an indication of the aggression of the agent. $\gamma$ is the weight given to the boost.
 However the aggression is capped to prevent the agent going berserk. The win is calculated as $\text {max}(0, 0.2 - \text{win rate})$. This caps the aggression of the agent to only when it has not won anything for long. If it winning at sufficient intervals, the agent does not need to overpay.

The weights $\alpha, \beta, \gamma$ are optimized using Optuna. This Optuna optimizes the bankroll received from our simulated auction environment, where our agent participates using the above bidding strategy. After tuning, we obtain the best values of the weights that maximize the bankroll.


In [ ]:
def objective(trial):
  alpha = trial.suggest_float("alpha",0.1,1.5)
  beta = trial.suggest_float("beta",0.0,0.5)
  gamma = trial.suggest_float("delta",0.0,0.05)

  env = CarAuctionEnv(df)

  trial_scores = []
  for game in range(100):
    obs, _ = env.reset()
    terminated = False

    while not terminated:
      price = obs[0]
      volatility = obs[1]
      bankroll = obs[2]
      round_num = obs[3]
      win_rate = obs[4]

      wins = max(0,0.2 - win_rate)
      base = price - (alpha * volatility)
      penalty = beta * (max(0,500000) - bankroll)/500000 * price
      boost = gamma * wins * price

      agent_bid = base - penalty + boost

      if agent_bid >= bankroll:
        agent_bid = bankroll

      obs, reward, terminated, truncated, info = env.step(agent_bid)

    trial_scores.append(env.bankroll)

  return np.mean(trial_scores)

In [ ]:
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials = 500, n_jobs=-1)

In [ ]:
#study.best_params:
#   ALPHA: 0.1007
#   BETA: 0.3792
#   GAMMA: 0.0125

# SIMULATING AUCTION FOR TEST

After tuning, I bring my agent into a simulation against 5 opponents - a normal distrbution, a fixed bidder ($0.9 * \text{price}$), a conservative bidder $(P_{10} + 50.0$) and a irrational bidder (bidding upto 98% of the price)

The agent participates against these opponents, with the cars drawn from random selection in the dataset. The agents participate in bidding, and the highest bid takes the car home. Each auction lasts for 500 cars(rounds).

After each auction, I track the bankroll, wins and win rate of my agent. I perform 10,000 simulations of this 500 round auction to obtain a very healthy data to test the generalizability of my agent. This constitutes a Monte Carlo Simulation.

After the simulation, I construct a 95% confidence interval to locate where my bankroll ends up after each 500 round auction. The obtained confidence interval is as follows:

| Category | Mean | 2.5%ile | 97.5%ile |
|----------| -----|---------|----------|
| Bankroll |522587|  486960 |  568090  |
| Wins     |102   |  55     |  128     |
| Win Rate |20%   |  11%    |  25%     |  

This shows that my agent is functioning rationally and that my bidding strategy helps my agent to win cars that are profitable, folds when someone else overpays or a bidding war emerges and sustains itself to preserve bankroll, netting profit 95% of the times while maintaining a healthy win rate and win counts. Thus my bidding strategy wins me healthy car counts and maintain profits maximizing both my win rate and my bankroll.

In [ ]:
class simulateAgents:

  def agent1Bid(self,price,volatility,bankroll,round_num,win_rate):
    alpha = 0.1007
    beta = 0.3792
    gamma = 0.0125

    win = max(0, 0.2-win_rate)
    base = price - (alpha * volatility)
    penalty = beta * max(0,(500000 - bankroll))/500000 * price
    boost = gamma * win * price

    bid = base - penalty + boost

    return bid

  def agent2Bid(self,price, volatility):
    bid = np.random.normal(loc=price, scale=abs(volatility))
    return max(0.0, bid)

  def agent3Bid(self,price):
    return price * 0.9

  def agent4Bid(self,p10):
      return p10 + 50.0

  def agent5Bid(self,price, current_bankroll):
    if current_bankroll > 400000:
        return price * 0.98
    elif current_bankroll > 200000:
        return price * 0.90
    else:
        return price * 0.75

In [ ]:
def runSimulation(df):
  bankroll = [500000.0] * 5
  wins = [0] * 5
  win_rate = [0.0] * 5
  round_num = 0

  agents = simulateAgents()
  sampled_df = df.sample(n=500, replace=False).reset_index(drop=True)
  for index in range(500):
      row = sampled_df.iloc[index]
      true_price = row['sellingprice']
      hammer_price = row['pred']
      volatility = row['std']
      p10 = row["quantile"]

      bid1 = agents.agent1Bid(hammer_price, volatility, bankroll[0], round_num, win_rate[0])
      bid2 = agents.agent2Bid(hammer_price, volatility)
      bid3 = agents.agent3Bid(hammer_price)
      bid4 = agents.agent4Bid(p10)
      bid5 = agents.agent5Bid(hammer_price,bankroll[4])

      bids = [bid1, bid2, bid3, bid4, bid5]

      sorted_bids = sorted(bids, reverse=True)
      highest_bid = sorted_bids[0]
      second_highest_bid = sorted_bids[1]

      price_paid = second_highest_bid + 50.0
      winner_index = bids.index(highest_bid)

      if price_paid <= bankroll[winner_index]:
          profit = true_price - price_paid
          bankroll[winner_index] += profit
          wins[winner_index] += 1

      for i in range(5):
          win_rate[i] = wins[i] / (round_num + 1)

      round_num += 1

  return bankroll, wins, win_rate

In [ ]:
result_bankroll = []
result_wins = []
result_win_rate = []

for sim in range(10000):

    final_bankrolls, final_wins, final_win_rate = runSimulation(df)

    result_bankroll.append(final_bankrolls)
    result_wins.append(final_wins)
    result_win_rate.append(final_win_rate)

result_bankroll = np.array(result_bankroll)
result_wins = np.array(result_wins)
result_win_rate = np.array(result_win_rate)

In [ ]:
agent1_bankroll = result_bankroll[:, 0]
agent1_wins = result_wins[:, 0]
agent1_win_rate = result_win_rate[:, 0]

mean_bankroll = np.mean(agent1_bankroll)
lower_bankroll = np.percentile(agent1_bankroll, 2.5)
upper_bankroll = np.percentile(agent1_bankroll, 97.5)

mean_wins = np.mean(agent1_wins)
lower_wins = np.percentile(agent1_wins, 2.5)
upper_wins = np.percentile(agent1_wins, 97.5)

mean_win_rate = np.mean(agent1_win_rate)
lower_win_rate = np.percentile(agent1_win_rate, 2.5)
upper_win_rate = np.percentile(agent1_win_rate, 97.5)

print("Agent 1:")

print("Mean:", mean_bankroll)
print("95% CI:", lower_bankroll, upper_bankroll)
print("Mean:", mean_wins)
print("95% CI:", lower_wins, upper_wins)
print("Mean:", mean_win_rate)
print("95% CI:", lower_win_rate, upper_win_rate)

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(agent1_bankroll, bins=20)
plt.axvline(lower_bankroll, color='red', linestyle='dashed', linewidth=2)
plt.axvline(upper_bankroll, color='red', linestyle='dashed', linewidth=2)
plt.axvline(mean_bankroll, color='green', linestyle='solid', linewidth=2)